# AI Incidents Pipeline — Google Colab

Este notebook corre el pipeline completo de análisis de incidentes de IA en Google Colab.

**Pasos:**
1. Ejecutá las celdas en orden de arriba a abajo
2. La única que **debés editar** es la celda de Configuración (paso 3)

> **Tip:** Si querés usar GPU para el análisis de sentimiento (más rápido), andá a `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU` antes de empezar.

## 1. Setup — clonar repositorio e instalar dependencias

Ejecutá esta celda **una sola vez** al inicio de cada sesión de Colab.

In [ ]:
import os

REPO_URL = "https://github.com/karenrg/incidents_pipeline"
REPO_DIR = "incidents_pipeline"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"El directorio '{REPO_DIR}' ya existe, saltando clone.")

%cd {REPO_DIR}

# Instala solo los paquetes que Colab no trae (evita conflictos de versiones)
!pip install -r requirements_colab.txt -q

print("\nSetup completo.")

## 2. Verificar entorno

In [ ]:
import torch

device = "GPU" if torch.cuda.is_available() else "CPU"
print(f"Dispositivo disponible: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Sin GPU. El paso de sentimiento tardará más en CPU (~5-10 min).")

## 3. Configuración

**Editá esta celda** según tu análisis. Todos los parámetros del pipeline están aquí.
Si no cambiás nada, corre con los valores por defecto del dataset OECD AIM.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DATOS
# ══════════════════════════════════════════════════════════════════════

# Path al CSV dentro del repositorio clonado.
# Si subís tu propio dataset, cambiá el nombre del archivo aquí.
SOURCE_PATH      = "data/raw/oecd_aim.csv"
PROCESSED_PATH   = "data/processed/incidents_processed.parquet"

# ══════════════════════════════════════════════════════════════════════
# FILTROS
# ══════════════════════════════════════════════════════════════════════

YEAR_RANGE = [2014, 2023]                              # Rango de años a analizar
REGIONS    = ["North America", "Europe", "Asia"]       # Regiones a incluir

# ══════════════════════════════════════════════════════════════════════
# NLP
# ══════════════════════════════════════════════════════════════════════

NLP_LANGUAGE       = "english"
MIN_TOKEN_LENGTH   = 3
CUSTOM_STOPWORDS   = ["ai", "u", "system", "report", "new", "say", "use", "would"]
MENTAL_HEALTH_KEYWORDS = ["mental health", "stress", "anxiety", "depression", "psychological"]

# ══════════════════════════════════════════════════════════════════════
# SENTIMIENTO
# ══════════════════════════════════════════════════════════════════════

SENTIMENT_BACKEND    = "transformer"   # Opciones: "transformer", "traditional_ml", "openai"
SENTIMENT_MODEL      = "cardiffnlp/twitter-roberta-base-sentiment-latest"  # Solo para backend=transformer
SENTIMENT_BATCH_SIZE = 32
THRESHOLD_HIGH       = 0.7             # Score >= 0.7 → "Alta"
THRESHOLD_MEDIUM     = 0.4             # Score >= 0.4 → "Media", < 0.4 → "Baja"
OPENAI_MODEL         = "gpt-4o-mini"   # Solo si backend="openai"
OPENAI_API_KEY       = ""              # Solo si backend="openai". Dejá vacío si no lo usás.

# ══════════════════════════════════════════════════════════════════════
# ANÁLISIS
# ══════════════════════════════════════════════════════════════════════

VULNERABLE_GROUPS      = ["Workers", "Children", "Minorities", "Women", "Migrants"]
TOP_N_PRINCIPLES       = 4
TOP_N_INDUSTRIES       = 10
TOP_N_HARMED           = 10
MOVING_AVG_WINDOWS     = [3, 6, 12]
HARM_CHRONOLOGY_TYPES  = ["Physical", "Psychological"]
EXCLUDE_VALUES         = ["Unknown"]

# ══════════════════════════════════════════════════════════════════════
# REPORTE
# ══════════════════════════════════════════════════════════════════════

REPORT_TITLE  = "Analisis de Incidentes de IA - OECD AIM"
FIGURES_DPI   = 300
RANDOM_STATE  = 42

# ══════════════════════════════════════════════════════════════════════
# Aplicar al config — no modificar esta sección
# ══════════════════════════════════════════════════════════════════════
import yaml, os

with open("config/params.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config["data"]["source_path"]                    = SOURCE_PATH
config["data"]["processed_path"]                 = PROCESSED_PATH
config["filters"]["year_range"]                  = YEAR_RANGE
config["filters"]["regions"]                     = REGIONS
config["nlp"]["language"]                        = NLP_LANGUAGE
config["nlp"]["min_token_length"]                = MIN_TOKEN_LENGTH
config["nlp"]["custom_stopwords"]                = CUSTOM_STOPWORDS
config["nlp"]["mental_health_keywords"]          = MENTAL_HEALTH_KEYWORDS
config["sentiment"]["backend"]                   = SENTIMENT_BACKEND
config["sentiment"]["model_name"]                = SENTIMENT_MODEL
config["sentiment"]["batch_size"]                = SENTIMENT_BATCH_SIZE
config["sentiment"]["thresholds"]["high"]        = THRESHOLD_HIGH
config["sentiment"]["thresholds"]["medium"]      = THRESHOLD_MEDIUM
config["sentiment"]["openai_model"]              = OPENAI_MODEL
config["analysis"]["vulnerable_groups"]          = VULNERABLE_GROUPS
config["analysis"]["top_n_principles"]           = TOP_N_PRINCIPLES
config["analysis"]["top_n_industries"]           = TOP_N_INDUSTRIES
config["analysis"]["top_n_harmed"]               = TOP_N_HARMED
config["analysis"]["moving_average_windows"]     = MOVING_AVG_WINDOWS
config["analysis"]["harm_chronology_types"]      = HARM_CHRONOLOGY_TYPES
config["analysis"]["exclude_values"]             = EXCLUDE_VALUES
config["reporting"]["report_title"]              = REPORT_TITLE
config["reporting"]["figures_dpi"]               = FIGURES_DPI
config["random_state"]                           = RANDOM_STATE

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("Configuración aplicada:")
print(f"  Dataset:    {SOURCE_PATH}")
print(f"  Años:       {YEAR_RANGE}")
print(f"  Regiones:   {REGIONS}")
print(f"  Sentimiento: {SENTIMENT_BACKEND}")

## 4. Imports e inicialización

In [ ]:
from pathlib import Path

from src import configure_logging, set_global_seeds
from src.ingestion import load_and_validate
from src.preprocessing import preprocess
from src.nlp import process_text
from src.sentiment import run_sentiment
from src.analysis import run_analysis
from src.visualization import run_visualization

configure_logging()
set_global_seeds(config["random_state"])
print("Listo.")

## 5. Ingestión

In [ ]:
df = load_and_validate(config)
df.head()

## 6. Preprocesamiento

In [ ]:
df = preprocess(df, config)
df.head()

## 7. NLP (tokenización y lematización)

In [ ]:
df = process_text(df, config)
df[["tokens", "mental_health_flag"]].head()

## 8. Análisis de sentimiento

> Con `backend=transformer`: descarga el modelo (~500 MB) la primera vez. Con GPU tarda ~2 min, sin GPU ~10 min.

In [ ]:
df = run_sentiment(df, config)
df[["sentiment_score", "sentiment_label"]].head()

## 9. Guardar dataset procesado

In [ ]:
processed_path = Path(config["data"]["processed_path"])
processed_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(processed_path)
print(f"Dataset guardado en: {processed_path}")

## 10. Análisis descriptivo

In [ ]:
metrics = run_analysis(df, config)

## 11. Visualización y reporte PDF

In [ ]:
report_path = run_visualization(df, metrics, config)
print(f"Reporte generado en: {report_path}")

## 12. Descargar outputs

In [ ]:
from google.colab import files

files.download(str(report_path))
files.download("outputs/reports/metrics.json")

print("Pipeline completado.")